# 0. ANCHOR Classical Planner Benchmark

ANCHOR Alpha is a deterministic, scientifically constrained research-and-education sandbox for investigating adaptive underwater-glider mission planning. It supports reproducible comparison of human, classical, and learning-based planners. It is not an operational ocean forecast or certified vehicle-navigation system.

**Plan. Simulate. Compare. Learn.**

This notebook proposes and visualizes plans outside the browser. It does not own mission validation, simulation, or official scoring.

Colab proposes. ANCHOR validates. ANCHOR simulates. ANCHOR scores.

Scientific boundary: these fixtures are deterministic synthetic benchmark data. They are suitable for reproducible planner comparison and classroom analysis, not calibrated ocean forecasting or certified navigation.

Fairness boundary: the default workflow is `FORECAST_ONLY`. Supported fairness classes are `FORECAST_ONLY`, `BELIEF_AWARE`, `PUBLIC_OBSERVATION_ONLY`, and explicit opt-in `ORACLE_HIDDEN_TRUTH`. Hidden truth is excluded unless an explicit oracle/debug workflow is selected and labeled.

Optimality boundary: An exact result is exact only for the stated candidate set, state representation, objective, and discretization. The Exact Bounded Small-Instance Oracle is bounded by declared candidates, objective, state representation, and discretization.

## 1. Configuration

Choose where the public benchmark data comes from and which transparent planners to run. The recommended Colab acceptance path is an uploaded public 4D benchmark bundle; local repository checkouts can use the checked-in bundle fixture.

In [ ]:
DATA_ACQUISITION_MODE = "bundle_upload"  # bundle_upload | checked_in_bundle | static_url | checked_in_fixture | upload
BENCHMARK_BUNDLE_PATH = "tests/fixtures/colab_benchmark/bundles/static_additive_routing.classical-planner-benchmark-bundle.json"
SOLVER_PACKET_PATH = "tests/fixtures/colab_benchmark/static_additive_routing_solver_packet.json"
STATIC_DOWNLOAD_BASE_URL = ""  # optional Pages/static base URL ending before tests/fixtures/...
STATIC_BUNDLE_PATH = BENCHMARK_BUNDLE_PATH
STATIC_SOLVER_PACKET_PATH = SOLVER_PACKET_PATH
BENCHMARK_FIXTURE_ID = "static_additive_routing"
OUTPUT_DIR = "anchor_benchmark_output"
PLANNER_SEED = 7
ACTIVE_GLIDER = "glider_01"
FAIRNESS_CLASS = "FORECAST_ONLY"
ALLOW_ORACLE_HIDDEN_TRUTH = False
ACCEPTANCE_MODE = True
CANDIDATE_NODE_LIMIT = 24
EXACT_ORACLE_SIZE_LIMIT = 6
REPEAT_COUNT = 3
TIMEOUT_PER_PLANNER_SECONDS = 5.0
PROFILE_POLICY = "mission/default"
PLANNERS = ["dijkstra", "astar", "weightedAstar", "greedyValuePerCost", "beamSearch", "timeExpandedAstar"]
RUN_EXACT_ORACLE = True
PLOT_FIGURES = True
RUN_NODE_REFEREE = False  # Colab execution packages remain pending until local ANCHOR finalization.

## 2. Environment Setup

COLAB ACCEPTANCE SETUP

The core planners use the Python standard library. `pandas` and `matplotlib` are optional notebook conveniences. Node is reported when present, but the notebook does not require Node to continue planner execution; local ANCHOR finalization remains authoritative.

In [ ]:
import json
import importlib.metadata as _version
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
TOOLS_PYTHON = REPO_ROOT / "tools" / "python"
if TOOLS_PYTHON.exists():
    sys.path.insert(0, str(TOOLS_PYTHON))

from anchor_benchmark import (
    astar_search,
    beam_search,
    build_anchor_plan,
    build_benchmark_record,
    build_bundle_parity_summary,
    build_colab_execution_package,
    build_colab_execution_report,
    build_parity_table,
    build_planning_problem,
    build_reproducibility_manifest,
    compare_results,
    dijkstra_search,
    exact_small_instance_oracle,
    extract_public_environment,
    greedy_value_per_cost,
    load_benchmark_bundle,
    load_json,
    reconstruct_public_bundle_arrays,
    run_planner_suite,
    sample_benchmark_bundle,
    sample_public_environment,
    solver_packet_from_benchmark_bundle,
    stable_digest,
    summarize_public_environment,
    time_expanded_astar,
    validate_benchmark_bundle,
    validate_bundle_parity_probes,
    validate_colab_execution_package,
    validate_parity_probes,
    validate_solver_packet,
    weighted_astar_search,
    write_json,
)
from anchor_benchmark.io import BENCHMARK_BOUNDARY, NOTEBOOK_VERSION, node_version, python_runtime_summary

def library_versions():
    versions = {}
    for name in ("numpy", "pandas", "matplotlib", "nbformat", "nbclient", "ipykernel", "jupyter_core", "jupyter_client", "nbconvert"):
        try:
            versions[name] = _version.version(name)
        except Exception:
            versions[name] = "unavailable"
    return versions

try:
    import pandas as pd
except Exception:
    pd = None
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
NODE_AVAILABLE = shutil.which("node") is not None
output_root = Path(OUTPUT_DIR)
for name in ["plans", "results", "benchmark_records", "figures", "tables", "packages"]:
    (output_root / name).mkdir(parents=True, exist_ok=True)

print("COLAB ACCEPTANCE SETUP")
print("Notebook version:", NOTEBOOK_VERSION)
print("Boundary:", BENCHMARK_BOUNDARY)
print("Acceptance mode:", ACCEPTANCE_MODE)
print("Data acquisition mode:", DATA_ACQUISITION_MODE)
print("Running in Colab:", IN_COLAB)
print("Python:", python_runtime_summary())
print("Node available:", NODE_AVAILABLE, node_version())
print("pandas:", getattr(pd, "__version__", "unavailable"))
print("matplotlib:", getattr(plt, "__version__", "available" if plt else "unavailable"))

## 3. Obtain Benchmark Data

Supported paths: upload an exported public 4D benchmark bundle, load a checked-in deterministic bundle fixture, fetch a public bundle from a configured static URL, or fall back to the legacy compact solver-packet representation. Bundle mode is preferred for Colab acceptance.

In [ ]:
from urllib.request import urlopen


def _download_json(relative_path, label):
    if not STATIC_DOWNLOAD_BASE_URL:
        raise ValueError("STATIC_DOWNLOAD_BASE_URL is required for static_url mode.")
    url = STATIC_DOWNLOAD_BASE_URL.rstrip("/") + "/" + relative_path.lstrip("/")
    print("Downloading", label, "from", url)
    with urlopen(url, timeout=30) as handle:
        return json.loads(handle.read().decode("utf-8"))


def _upload_json(expected_label):
    try:
        from google.colab import files
        uploaded = files.upload()
        first_name = next(iter(uploaded))
        print("Uploaded", expected_label, first_name)
        return json.loads(uploaded[first_name].decode("utf-8"))
    except Exception as exc:
        raise RuntimeError(f"{expected_label} upload requires Google Colab file upload support.") from exc


def load_benchmark_inputs():
    mode = DATA_ACQUISITION_MODE
    if mode == "bundle_upload":
        if IN_COLAB:
            bundle = _upload_json("benchmark bundle")
        elif Path(BENCHMARK_BUNDLE_PATH).exists():
            print("Local fallback: loading checked-in benchmark bundle", BENCHMARK_BUNDLE_PATH)
            bundle = load_json(BENCHMARK_BUNDLE_PATH)
        else:
            bundle = _upload_json("benchmark bundle")
        return bundle, solver_packet_from_benchmark_bundle(bundle), {"source": mode, "artifact": "benchmarkBundle"}
    if mode == "checked_in_bundle":
        bundle = load_benchmark_bundle(BENCHMARK_BUNDLE_PATH)
        return bundle, solver_packet_from_benchmark_bundle(bundle), {"source": mode, "artifact": "benchmarkBundle", "path": BENCHMARK_BUNDLE_PATH}
    if mode == "static_url":
        bundle = _download_json(STATIC_BUNDLE_PATH, "benchmark bundle")
        return bundle, solver_packet_from_benchmark_bundle(bundle), {"source": mode, "artifact": "benchmarkBundle", "path": STATIC_BUNDLE_PATH}
    if mode == "checked_in_fixture":
        packet = load_json(SOLVER_PACKET_PATH)
        return None, packet, {"source": mode, "artifact": "solverPacket", "path": SOLVER_PACKET_PATH}
    if mode == "upload":
        packet = _upload_json("solver packet")
        return None, packet, {"source": mode, "artifact": "solverPacket"}
    raise ValueError(f"Unknown DATA_ACQUISITION_MODE: {mode}")


benchmark_bundle, solver_packet, loaded_artifact = load_benchmark_inputs()
if benchmark_bundle is not None:
    print("Loaded benchmark bundle:", benchmark_bundle.get("bundleId"), "payload", benchmark_bundle.get("payloadDigest"))
    print("Bundle digest:", stable_digest(benchmark_bundle))
print("Loaded solver planning projection:", solver_packet.get("packetId"), "digest", stable_digest(solver_packet))
print("Loaded artifact:", loaded_artifact)

## 4. Validate and Inspect Artifacts

Hard failures stop benchmark execution. Bundle validation checks the public 4D transport, visibility, axes, shapes, field digests, and hidden-truth boundary. Solver-packet validation checks the planner-facing projection used by the classical graph builders.

In [ ]:
bundle_validation = None
if benchmark_bundle is not None:
    bundle_validation = validate_benchmark_bundle(benchmark_bundle, allow_hidden=ALLOW_ORACLE_HIDDEN_TRUTH)
    print(json.dumps(bundle_validation, indent=2))
    if bundle_validation["status"] == "FAIL":
        raise RuntimeError("Benchmark bundle validation failed before planning.")

validation = validate_solver_packet(solver_packet, allow_oracle=ALLOW_ORACLE_HIDDEN_TRUTH)
print(json.dumps(validation, indent=2))
if not validation["ok"]:
    raise RuntimeError("Solver packet validation failed before planning.")

artifact_overview = {
    "artifactKind": benchmark_bundle.get("type") if benchmark_bundle else solver_packet.get("type"),
    "schemaVersion": benchmark_bundle.get("schemaVersion") if benchmark_bundle else solver_packet.get("schemaVersion"),
    "benchmarkBundleDigest": (benchmark_bundle or {}).get("benchmarkBundleDigest") or (benchmark_bundle or {}).get("payloadDigest"),
    "publicProjectionDigest": (benchmark_bundle or {}).get("publicProjectionDigest") or solver_packet.get("publicProjectionDigest"),
    "solverPacketDigest": (benchmark_bundle or {}).get("solverPacketDigest") or validation["solverPacketDigest"],
    "environmentDigest": solver_packet.get("environmentDigest"),
    "missionDigest": solver_packet.get("missionDigest"),
    "coordinateFrame": (benchmark_bundle or {}).get("coordinateFrame") or solver_packet.get("coordinateFrame", "grid with cellSizeMeters"),
    "horizontalUnits": (benchmark_bundle or {}).get("horizontalUnits") or "meters via world.grid.cellSizeMeters",
    "depthConvention": (benchmark_bundle or {}).get("depthConvention") or "positive-down depth layers when present",
    "timeConvention": (benchmark_bundle or {}).get("timeConvention") or "seconds",
    "fieldRoles": (benchmark_bundle or {}).get("fieldRoles") or list(((solver_packet.get("planningData") or {}).get("visibleFields") or {}).keys()),
    "visibilityClass": (benchmark_bundle or {}).get("visibilityClass") or validation["visibilityClass"],
    "fairnessClass": (benchmark_bundle or {}).get("fairnessClass") or validation["fairnessClass"],
    "scoreProfileId": solver_packet.get("scoreProfileId"),
    "scoreProfileVersion": solver_packet.get("scoreProfileVersion"),
}
print(json.dumps(artifact_overview, indent=2))

## 5. Scientific Validation Context

The SCI-VALID-R2A manifest documents deterministic package evidence, claim boundaries, and limitations. It is not proof of operational ocean forecast accuracy.

In [ ]:
manifest_path = Path("validation/manifest.json")
if manifest_path.exists():
    validation_manifest = load_json(manifest_path)
    validation_context = {
        "manifestId": validation_manifest.get("manifestId") or validation_manifest.get("id"),
        "manifestDigest": validation_manifest.get("manifestDigest") or validation_manifest.get("digest"),
        "statusSummary": validation_manifest.get("statusSummary"),
        "evidenceLevelSummary": validation_manifest.get("evidenceLevelSummary"),
        "benchmarkSuitabilitySummary": validation_manifest.get("benchmarkSuitabilitySummary"),
    }
else:
    validation_context = {"manifestId": "UNKNOWN", "manifestDigest": solver_packet.get("validationBaselineDigest"), "statusSummary": "manifest file not found in this runtime"}
print(json.dumps(validation_context, indent=2))

## 6. Visualize the Environment

Visualizations use exported public benchmark data only by default. Bundle views inspect the declared source axes directly; they do not create new science fields and they are not authoritative parity evidence.

In [ ]:
problem = build_planning_problem(solver_packet, candidate_node_limit=CANDIDATE_NODE_LIMIT, profile_policy=PROFILE_POLICY)
print("Candidate count:", len(problem.candidates), "fairness:", problem.fairness_class)


def _selected_indices(count, limit=3):
    if count <= limit:
        return list(range(count))
    return sorted(set([0, count // 2, count - 1]))[:limit]


if benchmark_bundle is not None:
    east_axis = benchmark_bundle.get("eastAxisMeters") or []
    north_axis = benchmark_bundle.get("northAxisMeters") or []
    depths = benchmark_bundle.get("depthAxisMeters") or []
    times = benchmark_bundle.get("timeAxisSeconds") or []
    depth_indices = _selected_indices(len(depths), 3)
    time_indices = _selected_indices(len(times), 3)
    scalar_field = (benchmark_bundle.get("scalarFields") or [{}])[0]
    current_values = (benchmark_bundle.get("currents") or {}).get("values") or []
    scalar_values = scalar_field.get("values") or []
    bathymetry = (benchmark_bundle.get("bathymetry") or {}).get("bottomDepthMeters") or []
    print("Bundle visualization depth layers:", [depths[index] for index in depth_indices])
    print("Bundle visualization times:", [times[index] for index in time_indices])
    print("Scalar field:", scalar_field.get("fieldId"), scalar_field.get("units"), scalar_field.get("role"))
    profile_x = len(east_axis) // 2 if east_axis else 0
    profile_y = len(north_axis) // 2 if north_axis else 0
    profile_rows = []
    for depth_index in range(len(depths)):
        current_cell = current_values[0][depth_index][profile_y][profile_x]
        scalar_cell = scalar_values[0][depth_index][profile_y][profile_x]
        profile_rows.append({
            "eastMeters": east_axis[profile_x] if east_axis else None,
            "northMeters": north_axis[profile_y] if north_axis else None,
            "depthMeters": depths[depth_index],
            "uEastMetersPerSecond": current_cell[0],
            "vNorthMetersPerSecond": current_cell[1],
            "wDownMetersPerSecond": current_cell[2],
            "scalarValue": scalar_cell,
        })
    if pd is not None:
        display(pd.DataFrame(profile_rows))
    else:
        print("Vertical profile table:", json.dumps(profile_rows, indent=2))
    if PLOT_FIGURES and plt is not None:
        fig, ax = plt.subplots(figsize=(6, 4))
        image = ax.imshow(bathymetry, origin="lower", extent=[min(east_axis), max(east_axis), min(north_axis), max(north_axis)])
        ax.set_title("Exported public bathymetry, positive-down meters")
        ax.set_xlabel("East meters")
        ax.set_ylabel("North meters")
        fig.colorbar(image, ax=ax, label="bottom depth m")
        fig.savefig(output_root / "figures" / "bundle_bathymetry.png", dpi=140, bbox_inches="tight")
        plt.show()
        for time_index in time_indices:
            for depth_index in depth_indices:
                current_grid = current_values[time_index][depth_index]
                u = [[cell[0] for cell in row] for row in current_grid]
                v = [[cell[1] for cell in row] for row in current_grid]
                magnitude = [[(cell[0] ** 2 + cell[1] ** 2 + cell[2] ** 2) ** 0.5 for cell in row] for row in current_grid]
                scalar_grid = scalar_values[time_index][depth_index]
                fig, axes = plt.subplots(1, 2, figsize=(10, 4))
                axes[0].imshow(magnitude, origin="lower", extent=[min(east_axis), max(east_axis), min(north_axis), max(north_axis)])
                axes[0].quiver(east_axis, north_axis, u, v, color="white", scale=1.5)
                axes[0].set_title(f"Current magnitude/quiver t={times[time_index]}s z={depths[depth_index]}m")
                axes[1].imshow(scalar_grid, origin="lower", extent=[min(east_axis), max(east_axis), min(north_axis), max(north_axis)])
                axes[1].set_title(f"{scalar_field.get('fieldId')} t={times[time_index]}s z={depths[depth_index]}m")
                for axis in axes:
                    axis.set_xlabel("East meters")
                    axis.set_ylabel("North meters")
                fig.savefig(output_root / "figures" / f"bundle_fields_t{time_index}_z{depth_index}.png", dpi=140, bbox_inches="tight")
                plt.show()
    else:
        print("Bundle plotting skipped; matplotlib unavailable or PLOT_FIGURES=False.")
elif PLOT_FIGURES and plt is not None:
    from anchor_benchmark.visualization import plot_current_field, plot_environment_overview, plot_scalar_field
    figs = [plot_environment_overview(problem), plot_current_field(problem), plot_scalar_field(problem)]
    for index, fig in enumerate(figs, start=1):
        fig.savefig(output_root / "figures" / f"environment_{index:02d}.png", dpi=140, bbox_inches="tight")
        plt.show()
else:
    print("Plotting skipped; matplotlib unavailable or PLOT_FIGURES=False.")

## Exported Data Integrity and Web-App Parity

This section checks the public benchmark export that the notebook actually receives from ANCHOR. In bundle mode it validates the depth- and time-resolved public arrays directly: metadata, coordinate frame, axes, shapes, bathymetry, masks, current depths/times, scalar depths/times, mission geometry, numerical probes, field digests, and the public projection digest.

Visual agreement is an inspection aid. Canonical digests and numerical sample agreement are the authoritative parity evidence.

Do not compare notebook graphics and Three.js pixels directly. The checks below are numerical and metadata checks against the exported public data.

In [ ]:
if benchmark_bundle is not None:
    public_arrays = reconstruct_public_bundle_arrays(benchmark_bundle, prefer_numpy=True)
    public_summary = build_bundle_parity_summary(benchmark_bundle)
    parity_result = validate_bundle_parity_probes(benchmark_bundle)
    parity_table = parity_result["rows"]
    print("Benchmark bundle digest:", benchmark_bundle.get("benchmarkBundleDigest") or benchmark_bundle.get("payloadDigest"))
    print("Public projection digest:", benchmark_bundle.get("publicProjectionDigest"))
    print("Axis counts:", (bundle_validation or {}).get("axisCounts"))
    print("Data parity summary:", json.dumps(public_summary, indent=2))
    print("Parity probes:", parity_result["status"], parity_result["probeCount"], "failed", parity_result["failedProbeCount"])
    if parity_result["status"] == "FAIL":
        raise RuntimeError("Exported bundle parity probes failed; do not continue to planner comparison.")
    write_json(output_root / "tables" / "public_bundle_summary.json", public_summary)
    write_json(output_root / "tables" / "bundle_parity_probe_results.json", parity_result)
    write_json(output_root / "tables" / "public_bundle_arrays_manifest.json", {
        "keys": sorted(public_arrays.keys()),
        "numpyAvailable": "numpy" in public_arrays,
        "numpyUnavailable": public_arrays.get("numpyUnavailable"),
        "currentShape": (benchmark_bundle.get("currents") or {}).get("shape"),
        "scalarFields": [field.get("fieldId") for field in benchmark_bundle.get("scalarFields") or []],
    })
else:
    public_environment = extract_public_environment(solver_packet)
    public_summary = summarize_public_environment(solver_packet)
    parity_result = validate_parity_probes(solver_packet)
    parity_table = build_parity_table(solver_packet, parity_result)
    print("Public environment digest:", stable_digest(public_environment))
    print("Axes:", public_summary["axes"])
    print("Bathymetry status:", public_summary["bathymetry"])
    print("Mask counts:", public_summary["masks"])
    print("Current stats:", public_summary["currents"])
    print("Scalar stats:", public_summary["scalars"])
    print("Parity probes:", parity_result["status"], parity_result["probeCount"], "failed", parity_result["failedProbeCount"])
    if parity_result["status"] == "FAIL":
        raise RuntimeError("Exported-data parity probes failed; do not continue to planner comparison.")
    write_json(output_root / "tables" / "public_environment_summary.json", public_summary)

if pd is not None:
    display(pd.DataFrame(parity_table))
    display(pd.DataFrame(parity_result["rows"]))
else:
    print(json.dumps(parity_table, indent=2)[:4000])
    print(json.dumps(parity_result["rows"], indent=2)[:4000])

write_json(output_root / "tables" / "parity_probe_results.json", parity_result)
write_json(output_root / "tables" / "parity_table.json", {"rows": parity_table})

## 7. Construct the Planning Problem

Candidate discretization constrains the search space. Search costs guide plan construction; official mission outcomes come from ANCHOR simulation and scoring.

In [ ]:
cost_terms = [term.__dict__ for term in problem.cost_terms]
candidate_table = [node.__dict__ for node in problem.candidates]
print("Cost terms:")
print(json.dumps(cost_terms, indent=2))
if pd is not None:
    display(pd.DataFrame(candidate_table))
else:
    print(json.dumps(candidate_table[:8], indent=2))

## 8. Run Classical Planners

Dijkstra and A* can be exact on the declared graph under their assumptions. Weighted A*, Greedy Value per Predicted Cost, and Beam Search are labeled heuristic. Time-Expanded A* is exact only for declared time bins when the assumptions hold.

In [ ]:
planner_results = run_planner_suite(problem, PLANNERS)
if RUN_EXACT_ORACLE and len(problem.candidates) <= EXACT_ORACLE_SIZE_LIMIT + 1:
    planner_results.append(exact_small_instance_oracle(problem, candidate_limit=EXACT_ORACLE_SIZE_LIMIT, route_depth=4))

dijkstra = next((r for r in planner_results if r.planner_id == "dijkstra"), None)
astar = next((r for r in planner_results if r.planner_id == "astar"), None)
if dijkstra and astar:
    print("Dijkstra/A* cost delta:", abs(dijkstra.cost - astar.cost))

search_rows = compare_results(planner_results)
if pd is not None:
    display(pd.DataFrame(search_rows))
else:
    print(json.dumps(search_rows, indent=2))

## 9. Export Candidate ANCHOR Plans

Waypoints remain horizontal destinations. Incoming segment/profile metadata describes behavior used to reach the destination.

In [ ]:
plans = {}
for result in planner_results:
    plan = build_anchor_plan(problem, result, agent_id=ACTIVE_GLIDER)
    path = output_root / "plans" / f"{result.planner_id}.anchor.plan.json"
    write_json(path, plan)
    plans[result.planner_id] = {"path": path, "plan": plan, "result": result}
    print("wrote", path, stable_digest(plan))

## 10. Validate Plans with ANCHOR

Notebook checks are not enough. When Node and the repository are available, call the canonical ANCHOR plan validator.

In [ ]:
def run_node_validate(plan_path):
    if shutil.which("node") is None or not Path("tools/js/headless_validate_plan.mjs").exists():
        return {"ok": None, "status": "SKIPPED", "reason": "Node or ANCHOR repo files unavailable"}
    completed = subprocess.run(["node", "tools/js/headless_validate_plan.mjs", SOLVER_PACKET_PATH, str(plan_path)], capture_output=True, text=True)
    payload = json.loads(completed.stdout) if completed.stdout.strip().startswith("{") else {"stdout": completed.stdout, "stderr": completed.stderr}
    payload["returncode"] = completed.returncode
    return payload

validation_reports = {}
for planner_id, entry in plans.items():
    validation_reports[planner_id] = run_node_validate(entry["path"])
    print(planner_id, validation_reports[planner_id].get("ok"), validation_reports[planner_id].get("returncode"))

## 11. Simulate and Score with the Authoritative Referee

Every official benchmark score must originate from the same ANCHOR referee used by browser, headless, and benchmark exports. The Colab execution package remains PENDING_LOCAL_ANCHOR_REFEREE unless the returned package is finalized locally with ANCHOR Node packages.

In [ ]:
def run_node_referee(planner_id, plan_path):
    if not RUN_NODE_REFEREE or shutil.which("node") is None or not Path("tools/js/evaluate_colab_benchmark_plan.mjs").exists():
        return {"ok": None, "status": "SKIPPED", "reason": "Node referee unavailable"}
    out_dir = output_root / "results" / planner_id
    cmd = ["node", "tools/js/evaluate_colab_benchmark_plan.mjs", "--solver-packet", SOLVER_PACKET_PATH, "--plan", str(plan_path), "--out", str(out_dir), "--agent-id", ACTIVE_GLIDER]
    completed = subprocess.run(cmd, capture_output=True, text=True)
    payload = json.loads(completed.stdout) if completed.stdout.strip().startswith("{") else {"stdout": completed.stdout, "stderr": completed.stderr}
    payload["returncode"] = completed.returncode
    return payload

official_results = {}
for planner_id, entry in plans.items():
    official_results[planner_id] = run_node_referee(planner_id, entry["path"])
    print(planner_id, official_results[planner_id].get("ok"), official_results[planner_id].get("finalScore"))

## 12. Compare Algorithms

Tables keep planner solve time separate from ANCHOR validation/simulation/scoring time. Forecast-only and oracle-assisted results should not be silently mixed.

In [ ]:
comparison_rows = []
for row in search_rows:
    official = official_results.get(row["plannerId"], {})
    comparison_rows.append({**row, "officialScore": official.get("finalScore"), "evaluationOk": official.get("ok"), "benchmarkRecordDigest": official.get("benchmarkRecordDigest")})

summary_path = output_root / "tables" / "benchmark_results.csv"
if pd is not None:
    df = pd.DataFrame(comparison_rows)
    display(df)
    df.to_csv(summary_path, index=False)
else:
    import csv
    with summary_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison_rows[0].keys()))
        writer.writeheader()
        writer.writerows(comparison_rows)
print("wrote", summary_path)

## 13. Visualize Planned and Realized Outcomes

Planned routes are notebook artifacts. Realized trajectories, score results, and samples come from ANCHOR output bundles when available.

In [ ]:
if PLOT_FIGURES and plt is not None:
    from anchor_benchmark.visualization import plot_planner_routes
    fig = plot_planner_routes(problem, planner_results)
    fig.savefig(output_root / "figures" / "planner_routes.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("Route plot skipped.")

for planner_id, result in official_results.items():
    bundle = Path(result.get("files", {}).get("bundle", ""))
    if bundle.exists():
        print(planner_id, "bundle available:", bundle)

## 14. Export Benchmark Artifacts

The deterministic output structure is `anchor_benchmark_output/plans`, `results`, `benchmark_records`, `figures`, `tables`, plus summary and reproducibility files.

In [ ]:
records = []
for planner_id, entry in plans.items():
    official = official_results.get(planner_id, {})
    official_evaluation = {"officialScore": official.get("finalScore"), "scoreProfileId": official.get("scoreProfileId"), "scoreProfileVersion": official.get("scoreProfileVersion"), "scoreResultDigest": official.get("scoreResultDigest"), "simulationInputDigest": official.get("simulationInputDigest"), "simulationResultDigest": official.get("simulationResultDigest"), "totalEvaluationTimeSeconds": official.get("totalEvaluationTimeSeconds")}
    record = build_benchmark_record(problem, entry["result"], plan=entry["plan"], official_evaluation=official_evaluation)
    record_path = output_root / "benchmark_records" / f"{planner_id}.benchmark-record.json"
    write_json(record_path, record)
    records.append(record)

benchmark_summary = {
    "rows": comparison_rows,
    "solverPacketDigest": stable_digest(solver_packet),
    "benchmarkBundleDigest": (benchmark_bundle or {}).get("benchmarkBundleDigest") or (benchmark_bundle or {}).get("payloadDigest"),
    "publicProjectionDigest": (benchmark_bundle or {}).get("publicProjectionDigest") or solver_packet.get("publicProjectionDigest"),
    "boundary": BENCHMARK_BOUNDARY,
}
write_json(output_root / "benchmark_summary.json", benchmark_summary)
print("exported records:", len(records))

## 15. Reproducibility Summary

A planner's benchmark result is meaningful only with its environment digest, mission digest, fairness class, simulator version, scoring profile, public projection digest, benchmark bundle digest, and validation baseline.

The notebook emits a Colab execution report and Colab execution package. These are handoff artifacts for local authoritative finalization; they are not the final acceptance report.

In [ ]:
def git_commit():
    try:
        completed = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True)
        return completed.stdout.strip() or "UNKNOWN"
    except Exception:
        return "UNKNOWN"


def notebook_digest():
    notebook_path = Path("tools/python/notebooks/anchor_classical_planner_benchmark.ipynb")
    if notebook_path.exists():
        return stable_digest(json.loads(notebook_path.read_text(encoding="utf-8")))
    return stable_digest({"notebook": "anchor_classical_planner_benchmark", "version": NOTEBOOK_VERSION})


def result_summary(result):
    return {
        "completed": bool(result.route) and not result.timeout,
        "label": result.label,
        "optimalityStatus": result.optimality_status,
        "fairnessClass": result.fairness_class,
        "cost": result.cost,
        "visibleValue": result.value,
        "solveTimeSeconds": result.solve_time_seconds,
        "nodesExpanded": result.nodes_expanded,
        "edgesEvaluated": result.edges_evaluated,
        "maximumFrontierSize": result.maximum_frontier_size,
        "timeout": result.timeout,
    }


generated_paths = [str(path.relative_to(output_root)) for path in output_root.rglob("*") if path.is_file()]
manifest = build_reproducibility_manifest(problem, records, repository_commit=git_commit(), python_version=sys.version.split()[0], node_version=node_version(), generated_paths=generated_paths)
manifest["benchmarkBundleDigest"] = (benchmark_bundle or {}).get("benchmarkBundleDigest") or (benchmark_bundle or {}).get("payloadDigest")
manifest["publicProjectionDigest"] = (benchmark_bundle or {}).get("publicProjectionDigest") or solver_packet.get("publicProjectionDigest")
write_json(output_root / "reproducibility_manifest.json", manifest)

algorithms = {result.planner_id: result_summary(result) for result in planner_results}
dijkstra_result = algorithms.get("dijkstra")
astar_result = algorithms.get("astar")
if dijkstra_result and astar_result:
    algorithms["dijkstraAstarCostDelta"] = {"value": abs(float(dijkstra_result["cost"]) - float(astar_result["cost"])), "status": "PASS" if abs(float(dijkstra_result["cost"]) - float(astar_result["cost"])) <= 1e-6 else "WARN"}

execution_bundle = benchmark_bundle or {
    "benchmarkBundleDigest": validation["solverPacketDigest"],
    "payloadDigest": validation["solverPacketDigest"],
    "publicProjectionDigest": solver_packet.get("publicProjectionDigest") or validation["solverPacketDigest"],
    "environmentDigest": solver_packet.get("environmentDigest"),
    "missionDigest": solver_packet.get("missionDigest"),
    "solverPacketDigest": validation["solverPacketDigest"],
    "validationBaselineId": "scientific-validation-baseline-sci-valid-r2a",
    "validationBaselineDigest": solver_packet.get("validationBaselineDigest") or "fnv1a32:dd016175",
}

data_parity_summary = public_summary if benchmark_bundle is not None else {
    "schema": "PASS",
    "coordinates": "PASS",
    "axes": "PASS",
    "bathymetry": public_summary.get("bathymetry", "WARN") if isinstance(public_summary, dict) else "WARN",
    "masks": "PASS",
    "currents": "PASS",
    "currentDepths": "PASS",
    "currentTimes": "PASS",
    "scalars": "PASS",
    "scalarDepths": "PASS",
    "scalarTimes": "PASS",
    "missionGeometry": "PASS",
    "fieldDigests": "PASS",
    "publicProjectionDigest": execution_bundle.get("publicProjectionDigest"),
    "probeCount": parity_result.get("probeCount", 0),
    "failedProbeCount": parity_result.get("failedProbeCount", 0),
}

plans_for_package = []
for planner_id, entry in plans.items():
    plans_for_package.append({
        "plannerId": planner_id,
        "plannerClass": "classical",
        "fairnessClass": entry["plan"].get("fairnessClass") or FAIRNESS_CLASS,
        "optimalityStatus": entry["result"].optimality_status,
        "plan": entry["plan"],
    })

execution_report = build_colab_execution_report(
    bundle=execution_bundle,
    parity_result=data_parity_summary,
    algorithms=algorithms,
    exported_plan_digests=[stable_digest(entry["plan"]) for entry in plans.values()],
    notebook_digest=notebook_digest(),
    repository_commit=git_commit(),
    official_evaluation_status="PENDING_LOCAL_ANCHOR_REFEREE",
    warnings=[] if benchmark_bundle is not None else ["Legacy compact solver-packet fallback used; public 4D bundle upload is preferred for acceptance."],
    failures=[] if parity_result.get("failedProbeCount", 0) == 0 else ["Parity probes failed."],
    library_versions=library_versions(),
)
execution_package = build_colab_execution_package(
    execution_report=execution_report,
    bundle=execution_bundle,
    plans=plans_for_package,
    planner_metrics=algorithms,
    parity_probe_results=parity_result,
    reproducibility_manifest=manifest,
)
package_validation = validate_colab_execution_package(execution_package)
if package_validation["status"] == "FAIL":
    execution_report["status"] = "FAIL"
    execution_report["failures"] = execution_report.get("failures", []) + package_validation["failures"]
    execution_report["reportDigest"] = stable_digest({key: value for key, value in execution_report.items() if key != "reportDigest"})
    execution_package = build_colab_execution_package(
        execution_report=execution_report,
        bundle=execution_bundle,
        plans=plans_for_package,
        planner_metrics=algorithms,
        parity_probe_results=parity_result,
        reproducibility_manifest=manifest,
    )

write_json(output_root / "colab_execution_report.json", execution_report)
write_json(output_root / "colab_execution_package.json", execution_package)
write_json(output_root / "packages" / "colab_execution_package.json", execution_package)
print(json.dumps(manifest, indent=2)[:4000])
print(json.dumps({"executionStatus": execution_report["status"], "officialEvaluationStatus": execution_report["officialEvaluationStatus"], "reportDigest": execution_report["reportDigest"], "packageDigest": execution_package["packageDigest"], "packageValidation": package_validation}, indent=2))
if execution_report["status"] == "PASS" and package_validation["status"] == "PASS":
    print("COLAB EXECUTION: PASS")
else:
    print("COLAB EXECUTION: FAIL")
print("Local finalization command:")
print("node tools/js/finalize_colab_benchmark_acceptance.mjs anchor_benchmark_output/colab_execution_package.json")
print("npm.cmd run validate:colab-acceptance -- anchor_benchmark_output/colab_acceptance_report.json")

try:
    from google.colab import files
    print("In Colab: download colab_execution_report.json, colab_execution_package.json, and reproducibility_manifest.json from", output_root)
except Exception:
    print("Outside Colab, output artifacts are available under", output_root)